# Zerlegung der Schadenregulierungs-Variabilität über die Organisationshierarchie eines Versicherers mit PROC NESTED


## Zusammenfassung

Ein Sach- und Unfallversicherer möchte wissen, **wo** die Uneinheitlichkeit
bei Schadenregulierungen ihren Ursprung hat: Wird sie durch Unterschiede
zwischen geografischen Regionen, zwischen Filialen, zwischen einzelnen
Sachbearbeitern oder einfach durch Rauschen von Fall zu Fall verursacht?
Dieses Notebook erzeugt einen synthetischen, vollständig geschachtelten
Bestand an Kfz-Schadendaten (Region > Filiale > Sachbearbeiter > Schadenfall)
und verwendet **PROC NESTED**, um eine Zufallseffekte-Varianzanalyse
durchzuführen, wobei die von jeder Hierarchieebene beigetragene
Varianzkomponente geschätzt und jeweils als Prozentsatz der Gesamtvarianz
ausgewiesen wird. Das Ergebnis sagt dem Management, auf welche
Organisationsebene es zur Verbesserung der Regulierungskonsistenz zielen
sollte.


## Datenquellen

Alle Daten werden inline durch den ersten DATA-Schritt erzeugt — keine externen Dateien, kein Netzwerk. Das Design ist **vollständig geschachtelt**: jede Filiale gehört zu genau einer Region, jeder Sachbearbeiter zu genau einer Filiale, und jeder Schadenfall zu genau einem Sachbearbeiter.

| Datensatz | Zeilen | Variable | Typ | Rolle | Beschreibung |
|---------|------|----------|------|------|-------------|
| `claims` | 40 | `Region` | char | CLASS (Ebene 1, äußerste) | Geografische Region (5 Regionen: Nordost, Südost, Mitte, West, Südwest) |
| | | `Branch` | char | CLASS (Ebene 2, geschachtelt in Region) | Filialcode (2 Filialen je Region) |
| | | `Adjuster` | char | CLASS (Ebene 3, geschachtelt in Branch) | Sachbearbeiter-ID (2 Sachbearbeiter je Filiale) |
| | | `Settlement` | num | VAR (abhängig) | Für den Kfz-Schaden gezahlte Entschädigung, in USD |
| | | `CycleDays` | num | VAR (abhängig) | Tage vom Erstschadenmeldedatum bis zur Regulierung |

Synthetische Struktur: 5 Regionen x 2 Filialen x 2 Sachbearbeiter x 2 Schadenfälle = 40 Beobachtungen. Ein Regions-Zufallseffekt, ein Filiale-innerhalb-Region-Zufallseffekt, ein Sachbearbeiter-innerhalb-Filiale-Zufallseffekt und ein Residuum auf Schadenfall-Ebene werden additiv über `rand('NORMAL', ...)` überlagert, sodass die Varianzkomponenten rekonstruierbar sind. Der Regionseffekt wird mit der größten Standardabweichung (2.200) gezogen, sodass *wo* ein Schaden bearbeitet wird der dominierende Treiber ist. Startwert fixiert mit `call streaminit(20260531)`. Ein kompaktes, vollständig balanciertes Design hält jede Ebene der Hierarchie mit echten Freiheitsgraden besetzt.


# Zerlegung der Schadenregulierungs-Variabilität mit PROC NESTED

Wenn ein Sach- und Unfallversicherer prüft, wie viel er für die Regulierung von Kfz-Schäden zahlt, stellt er oft eine große Streuung fest. Ein Teil dieser Streuung ist unvermeidlich (jeder Unfall ist anders), aber ein Teil spiegelt **organisatorische** Faktoren wider — eine Region zahlt großzügiger als eine andere, eine Filiale ist nachgiebiger, ein einzelner Sachbearbeiter ist ein Ausreißer.

Die Daten haben eine streng **geschachtelte** (hierarchische) Struktur:

```
Region  ->  Filiale (geschachtelt in Region)  ->  Sachbearbeiter (geschachtelt in Filiale)  ->  einzelne Schadenfälle
```

Eine Filiale gehört zu genau einer Region und ein Sachbearbeiter zu genau einer Filiale, sodass die Faktoren geschachtelt statt gekreuzt sind. `PROC NESTED` führt für genau dieses Design eine Zufallseffekte-Varianzanalyse durch und schätzt auf jeder Ebene eine **Varianzkomponente**, ausgedrückt als Prozentsatz der Gesamtvarianz. Diese prozentuale Aufschlüsselung beantwortet die geschäftliche Frage: *auf welche Organisationsebene sollten wir zielen, um Regulierungen konsistenter zu machen?*


## Schritt 1 — Erzeugung eines synthetischen, vollständig geschachtelten Schadenbestands

Wir simulieren 5 Regionen, jede mit 2 Filialen, jede mit 2 Sachbearbeitern, jeder bearbeitet 2 Schadenfälle (insgesamt 40 Zeilen). Wir bauen die Zielgröße aus additiven Zufallseffekten auf, sodass jede Ebene tatsächlich zur Varianz beiträgt:

- ein **Region**-Effekt (größte Streuung — Regionen unterscheiden sich am meisten),
- ein **Filiale-innerhalb-Region**-Effekt,
- ein **Sachbearbeiter-innerhalb-Filiale**-Effekt,
- und ein **Residuum auf Schadenfall-Ebene** (das Rauschen innerhalb eines Sachbearbeiters).

`call streaminit` fixiert den Startwert für Reproduzierbarkeit; jeder Effekt wird mit `rand('NORMAL', mean, sd)` gezogen. Der Regionseffekt verwendet die größte Standardabweichung, sodass er den größten Anteil der Gesamtvarianz beanspruchen sollte. Wir erzeugen außerdem eine zweite Zielgröße, `CycleDays`, die dieselbe Hierarchie teilt, sodass wir später die Mehrfach-Zielgrößen-Analyse demonstrieren können.


In [1]:
DATEN claims;
   AUFRUFEN streaminit(20260531);
   LÄNGE Region $10 Branch $14 Adjuster $18;

   /* Regionsebene-Zufallseffekt: Regionen unterscheiden sich am staerksten. */
   AUSFÜHRUNG r = 1 BIS 5;
      Region = SCAN('Nordost Suedost Mitte West Suedwest', r);
      RegionEffect  = rand('NORMAL', 0, 2200);
      RegionCycle   = rand('NORMAL', 0, 11);

      /* Filiale, geschachtelt innerhalb der Region. */
      AUSFÜHRUNG b = 1 BIS 2;
         Branch = catx('-', Region, PUT(b, z2.));
         BranchEffect = rand('NORMAL', 0, 700);
         BranchCycle  = rand('NORMAL', 0, 4);

         /* Sachbearbeiter, geschachtelt innerhalb der Filiale. */
         AUSFÜHRUNG a = 1 BIS 2;
            Adjuster = catx('-', Branch, PUT(a, z1.));
            AdjEffect = rand('NORMAL', 0, 450);
            AdjCycle  = rand('NORMAL', 0, 2.5);

            /* Einzelne Schadensfaelle, bearbeitet von diesem Sachbearbeiter. */
            AUSFÜHRUNG claim = 1 BIS 2;
               Settlement = 8500
                          + RegionEffect + BranchEffect + AdjEffect
                          + rand('NORMAL', 0, 1100);
               CycleDays  = 21
                          + RegionCycle + BranchCycle + AdjCycle
                          + rand('NORMAL', 0, 6);
               WENN Settlement < 0 DANN Settlement = 0;
               WENN CycleDays  < 1 DANN CycleDays  = 1;
               AUSGABE;
            ENDE;
         ENDE;
      ENDE;
   ENDE;

   BEHALTEN Region Branch Adjuster Settlement CycleDays;
AUSFÜHREN;



NOTE: DATA claims


NOTE: Wrote claims (40 rows, 5 columns).
NOTE: DATA elapsed:
  wall  0.00 seconds
  cpu   0.00 seconds


## Schritt 2 — Sortierung nach der Schachtelungshierarchie

`PROC NESTED` erfordert, dass die Eingabedaten **nach den CLASS-Variablen in der Reihenfolge sortiert sind, in der sie aufgeführt werden** — der äußerste Faktor zuerst. Wir sortieren nach `Region Branch Adjuster`, damit die Prozedur die Hierarchie korrekt durchlaufen kann.


In [2]:
PROC SORT DATEN=claims;
   NACH Region Branch Adjuster;
AUSFÜHREN;



NOTE: PROC SORT data=claims

NOTE: Unlicensed mode - input limited to 100 observations.
NOTE: Read 40 rows from claims.
NOTE: Wrote claims (40 rows, 5 columns).
NOTE: PROC SORT statement used.


## Schritt 3 — Varianzkomponentenanalyse des Schadenbetrags

Die Kernanalyse. Wir führen die CLASS-Variablen von außen nach innen auf (`Region Branch Adjuster`); die innerste Wiederholung — die einzelnen Schadenfälle — wird automatisch als Fehlerterm behandelt. `VAR Settlement` benennt die einzige Zielgröße.

Die `LABEL`-Anweisung liefert einen verständlicheren Anzeigenamen für die Zielgröße im Ausgabebanner. Die Ausgabe erzeugt die Koeffizienten der erwarteten mittleren Quadrate, die Varianzanalyse-Tabelle (mit einem F-Test jeder Varianzkomponente gegen null) und die Schätzung der **Varianzkomponente** auf jeder Ebene zusammen mit ihrem **Prozentsatz der Gesamtvarianz**.


In [3]:
TITLE1 'Varianzkomponenten der Kfz-Schadenregulierungen';
TITLE2 'Zufallseffekte-ANOVA nach Region / Filiale / Sachbearbeiter';

PROZEDUR nested DATEN=claims;
   KLASSE Region Branch Adjuster;
   VAR Settlement;
   BEZEICHNUNG Settlement = 'Schadenzahlung (USD)';
AUSFÜHREN;


                                    Varianzkomponenten der Kfz-Schadenregulierungen                                     
                              Zufallseffekte-ANOVA nach Region / Filiale / Sachbearbeiter                               


                       Nested Random Effects Analysis of Variance

Nested Random Effects Analysis of Variance for Variable Schadenzahlung (USD)
Variance Source       DF  Sum of Squares       F Value      Pr > F  Error Term   Mean Square  Variance Component    Percent of Total    EMS Coef
Region                 4  62114122.126866          6.71      0.0304    Branch  15528530.531717  1651824.844989             54.4057      8.0000
Branch                 5  11569658.859028          1.89      0.1829  Adjuster  2313931.771806   272682.828942              8.9813      4.0000
Adjuster              10  12232004.560376          1.22      0.3348     Error  1223200.456038   111582.782346              3.6752      2.0000
Error                 20  20000697.82690


NOTE: Option TITLE changed to Varianzkomponenten der Kfz-Schadenregulierungen.
NOTE: Option TITLE2 changed to Zufallseffekte-ANOVA nach Region / Filiale / Sachbearbeiter.
NOTE: PROC NESTED nested ANOVA / variance components

NOTE: PROC NESTED statement used.


## Schritt 4 — Zwei Zielgrößen gleichzeitig analysieren

Das Management interessiert sich auch für die **Bearbeitungsdauer** — wie lange es dauert, bis Schadenfälle reguliert werden. Wir fügen `CycleDays` der `VAR`-Liste hinzu. Bei mehr als einer abhängigen Variable meldet `PROC NESTED` zusätzlich eine **Kovariationsanalyse**, die zeigt, wie die beiden Zielgrößen auf jeder Hierarchieebene mitvariieren (zum Beispiel, ob die Regionen, die mehr zahlen, auch langsamer regulieren).


In [4]:
TITLE1 'Varianzkomponenten von Schadenhoehe und Bearbeitungsdauer';
TITLE2 'Zwei Zielgroessen in derselben geschachtelten Hierarchie';

PROZEDUR nested DATEN=claims;
   KLASSE Region Branch Adjuster;
   VAR Settlement CycleDays;
   BEZEICHNUNG Settlement = 'Schadenzahlung (USD)'
         CycleDays  = 'Tage bis zur Regulierung';
AUSFÜHREN;


                               Varianzkomponenten von Schadenhoehe und Bearbeitungsdauer                                
                                Zwei Zielgroessen in derselben geschachtelten Hierarchie                                


                       Nested Random Effects Analysis of Variance

Nested Random Effects Analysis of Variance for Variable Schadenzahlung (USD)
Variance Source       DF  Sum of Squares       F Value      Pr > F  Error Term   Mean Square  Variance Component    Percent of Total    EMS Coef
Region                 4  62114122.126866          6.71      0.0304    Branch  15528530.531717  1651824.844989             54.4057      8.0000
Branch                 5  11569658.859028          1.89      0.1829  Adjuster  2313931.771806   272682.828942              8.9813      4.0000
Adjuster              10  12232004.560376          1.22      0.3348     Error  1223200.456038   111582.782346              3.6752      2.0000
Error                 20  20000697.82690


NOTE: Option TITLE changed to Varianzkomponenten von Schadenhoehe und Bearbeitungsdauer.
NOTE: Option TITLE2 changed to Zwei Zielgroessen in derselben geschachtelten Hierarchie.
NOTE: PROC NESTED nested ANOVA / variance components

NOTE: PROC NESTED statement used.


## Schritt 5 — Reine Varianzansicht mit der AOV-Option

Für eine schnelle Zusammenfassung der Varianzkomponenten über beide Zielgrößen beschränkt die Option `AOV` die Ausgabe auf die Varianzanalyse-Statistiken und **unterdrückt den Abschnitt der Kovariationsanalyse**. Dies ist die kompakte Ansicht, die ein Portfolioanalyst durchsehen würde, wenn er nur die Varianzaufschlüsselung je Ebene für jede Zielgröße braucht, nicht die zielgrößenübergreifende Kovariation.


In [5]:
TITLE1 'Nur Varianzkomponenten (AOV)';

PROZEDUR nested DATEN=claims aov;
   KLASSE Region Branch Adjuster;
   VAR Settlement CycleDays;
   BEZEICHNUNG Settlement = 'Schadenzahlung (USD)'
         CycleDays  = 'Tage bis zur Regulierung';
AUSFÜHREN;

TITEL;


                                              Nur Varianzkomponenten (AOV)                                              
                                Zwei Zielgroessen in derselben geschachtelten Hierarchie                                


                       Nested Random Effects Analysis of Variance

Nested Random Effects Analysis of Variance for Variable Schadenzahlung (USD)
Variance Source       DF  Sum of Squares       F Value      Pr > F  Error Term   Mean Square  Variance Component    Percent of Total    EMS Coef
Region                 4  62114122.126866          6.71      0.0304    Branch  15528530.531717  1651824.844989             54.4057      8.0000
Branch                 5  11569658.859028          1.89      0.1829  Adjuster  2313931.771806   272682.828942              8.9813      4.0000
Adjuster              10  12232004.560376          1.22      0.3348     Error  1223200.456038   111582.782346              3.6752      2.0000
Error                 20  20000697.82690


NOTE: Option TITLE changed to Nur Varianzkomponenten (AOV).
NOTE: PROC NESTED nested ANOVA / variance components

NOTE: PROC NESTED statement used.


## Interpretation der Ergebnisse

Die Spalte **Prozent der Gesamtvarianz** in der Varianzanalyse-Tabelle ist die Kernaussage. Liest man sie von oben nach unten, erhält man den Anteil der gesamten Schadenregulierungs-Variabilität, der jeder Organisationsebene zuzuschreiben ist. Für den Schadenbetrag liefert der Lauf:

- **Region — 54,4 %** (Pr > F = 0,0304). Die Daten wurden mit der größten Streuung auf Regionsebene erzeugt, und die Analyse deckt dies auf: Mehr als die Hälfte der gesamten Schadenregulierungs-Variabilität liegt *zwischen* den Regionen — statistisch signifikante Evidenz dafür, dass *wo* ein Schaden bearbeitet wird, der dominierende Treiber ist.
- **Filiale (innerhalb Region) — 9,0 %** (Pr > F = 0,18). Ein moderater zusätzlicher Anteil, sobald man von der Region auf die einzelne Filiale herunterbricht; hier nicht signifikant.
- **Sachbearbeiter (innerhalb Filiale) — 3,7 %** (Pr > F = 0,33). In diesem Schadenbestand ist kaum Variation auf den einzelnen Sachbearbeiter zurückzuführen.
- **Fehler — 32,9 %.** Das residuale Rauschen von Schadenfall zu Schadenfall innerhalb eines Sachbearbeiters; dies ist die irreduzible Variation, die kein organisatorischer Hebel beseitigen kann.

Jede Ebene trägt außerdem einen **F-Test (Pr > F)** der Nullhypothese, dass ihre Varianzkomponente null ist. Der kleine p-Wert für Region (0,0304) ist ein statistischer Beleg für echte systematische Unterschiede zwischen den Regionen, nicht für Stichprobenzufall.

Die Zielgröße Bearbeitungsdauer erzählt eine parallele Geschichte: **Region macht 45,8 %** der Variation der Tage-bis-zur-Regulierung aus (Pr > F = 0,0448, ebenfalls signifikant), wobei Filiale und Sachbearbeiter einstellige Anteile beisteuern und das Residuum 40,1 % trägt. Sowohl *wie viel* ein Versicherer zahlt als auch *wie lange* es dauert, werden also in erster Linie von der Region bestimmt.

Der Lauf mit zwei Zielgrößen fügt eine **Kovariationsanalyse** hinzu. Hier treibt die Ebene Region die Kreuzprodukte an, und der gesamte **Korrelationskoeffizient beträgt -0,36** — ein negativer Zusammenhang: Die Regionen, die höhere Schadenzahlungen leisten, tendieren dazu, diese *schneller* abzuschließen, nicht langsamer. Das ist ein nützlicher, nicht offensichtlicher Befund — die teuren Regionen sind nicht die langsamen.

**Geschäftliche Erkenntnis:** Da Region die Prozent-der-Gesamtvarianz-Aufschlüsselung für beide Zielgrößen dominiert, sollte der Versicherer zunächst Regulierungsrichtlinien und Vollmachtsgrenzen *regionsübergreifend* vereinheitlichen — dort liegt die größte, statistisch signifikante Uneinheitlichkeit — bevor er in Maßnahmen auf Filial- oder Sachbearbeiterebene investiert. Die negative Korrelation zwischen Schadenzahlung und Bearbeitungsdauer bedeutet, dass es keine einzelne Region gibt, die sowohl am teuersten als auch am langsamsten ist, sodass die beiden Probleme mit getrennten, regionsbezogenen Programmen angegangen werden können. PROC NESTED verwandelt ein vages Gefühl von "unsere Regulierungen sind uneinheitlich" in eine handlungsfähige, ebenenweise Zuordnung, wo diese Uneinheitlichkeit ihren Ursprung hat.
